# DỰ ĐOÁN BỆNH TIỂU ĐƯỜNG — EDA & PREPROCESSING

**Assignment 03 — Neural Networks and Representation Learning**

**Bài toán:** Phân loại nhị phân — dự đoán nguy cơ tiểu đường

**Môi trường:** conda env `assignment2`

---

## 1. Mục tiêu notebook

Notebook này thực hiện:
1. Tải và khám phá dữ liệu (`diabetes_prediction_dataset.csv`)
2. Kiểm tra chất lượng dữ liệu: missing value, duplicates, outliers
3. Phân tích phân bố target và tương quan
4. **Tiền xử lý**: mã hoá biến categorical, xử lý mất cân bằng lớp, chuẩn hoá
5. **Chia tập**: train / validation / test

Nội dung notebook phục vụ cho các notebook tiếp theo:
- `2_ml_models.ipynb` — 3 mô hình ML cơ bản
- `3_deep_learning.ipynb` — Deep Learning (NumPy từ đầu + MLP PyTorch)
- `4_ml_vs_dl_comparison.ipynb` — So sánh 4 mô hình

## 2. Import thư viện

In [2]:
import matplotlib
matplotlib.use('Agg')  # Fix no DISPLAY error in headless / notebook environments
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cấu hình đồ thị
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = os.path.join('..', 'data', 'diabetes_prediction_dataset.csv')
print('Ready.')

Ready.


## 3. Tải dữ liệu

In [3]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

Shape: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


### 3.1. Kiểu dữ liệu & thông tin cơ bản

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  str    
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  str    
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 6.9 MB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,100000.0,41.885856,22.516840,0.08,24.00,43.00,60.00,80.00
hypertension,100000.0,0.074850,0.263150,0.00,0.00,0.00,0.00,1.00
heart_disease,100000.0,0.039420,0.194593,0.00,0.00,0.00,0.00,1.00
bmi,100000.0,27.320767,6.636783,10.01,23.63,27.32,29.58,95.69
HbA1c_level,100000.0,5.527507,1.070672,3.50,4.80,5.80,6.20,9.00
blood_glucose_level,100000.0,138.058060,40.708136,80.00,100.00,140.00,159.00,300.00
diabetes,100000.0,0.085000,0.278883,0.00,0.00,0.00,0.00,1.00


### 3.2. Kiểm tra missing values

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('✅ Không có missing values.')
else:
    print('❌ Có missing values:')
    print(missing)

✅ Không có missing values.


### 3.3. Kiểm tra duplicates

In [7]:
dup = df.duplicated().sum()
print(f'🔁 Số dòng trùng lặp: {dup} ({dup/len(df)*100:.2f}%)')
print('\nSố lượng mỗi nhóm target trước khi xử lý:')
print(df['diabetes'].value_counts())

🔁 Số dòng trùng lặp: 3854 (3.85%)

Số lượng mỗi nhóm target trước khi xử lý:
diabetes
0    91500
1     8500
Name: count, dtype: int64


### 3.4. Xử lý duplicates

Loại bỏ các dòng trùng lặp hoàn toàn để tránh nhiễu cho quá trình huấn luyện.

In [8]:
df = df.drop_duplicates().reset_index(drop=True)
print('Shape sau khi loại duplicate:', df.shape)
print('\nPhân bố target sau khi loại duplicate:')
print(df['diabetes'].value_counts())
print(f"\nTỷ lệ lớp 1 (diabetes): {df['diabetes'].mean()*100:.2f}%")

Shape sau khi loại duplicate: (96146, 9)

Phân bố target sau khi loại duplicate:
diabetes
0    87664
1     8482
Name: count, dtype: int64

Tỷ lệ lớp 1 (diabetes): 8.82%


## 4. Phân tích đơn biến (Univariate Analysis)

### 4.1. Phân bố target (diabetes)

In [9]:
ax = df['diabetes'].value_counts().plot(kind='bar', color=['#1f77b4','#d62728'])
ax.set_title('Phân bố biến target (diabetes)')
ax.set_xticklabels(['Không tiểu đường (0)', 'Tiểu đường (1)'], rotation=0)
ax.set_ylabel('Số lượng')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()+500), ha='center')
plt.tight_layout(); plt.show()

### 4.2. Phân bố các biến số

In [10]:
num_cols = ['age','bmi','HbA1c_level','blood_glucose_level']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'Phân bố {col}')
plt.tight_layout(); plt.show()

### 4.3. Phân bố biến categorical

**Giới tính (`gender`)** và **tiền sử hút thuốc (`smoking_history`)**.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='gender', ax=axes[0])
axes[0].set_title('Phân bố gender')
sns.countplot(data=df, x='smoking_history', order=df['smoking_history'].value_counts().index, ax=axes[1])
axes[1].set_title('Phân bố smoking_history')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## 5. Phân tích nhị biến (Bivariate Analysis)

### 5.1. Ma trận tương quan giữa biến số và target

Hệ số tương quan Pearson cho thấy mức độ liên quan tuyến tính giữa các biến số và target.

In [12]:
# Chỉ tính các biến số
df_numeric = df.select_dtypes(include=[np.number])
corr = df_numeric.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Ma trận tương quan')
plt.tight_layout(); plt.show()

### 5.2. Tương quan với target

In [13]:
corr_with_target = corr['diabetes'].drop('diabetes').sort_values(ascending=False)
print('Tương quan tuyến tính của các biến với target (diabetes):')
print(corr_with_target.round(3).to_string())

Tương quan tuyến tính của các biến với target (diabetes):
blood_glucose_level    0.424
HbA1c_level            0.406
age                    0.265
bmi                    0.215
hypertension           0.196
heart_disease          0.171


### 5.3. Tỷ lệ tiểu đường theo từng nhóm

In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.barplot(data=df, x='gender', y='diabetes', ax=axes[0,0])
axes[0,0].set_title('Tỷ lệ tiểu đường theo gender')

sns.barplot(data=df, x='hypertension', y='diabetes', ax=axes[0,1])
axes[0,1].set_title('Tỷ lệ tiểu đường theo hypertension')

sns.barplot(data=df, x='heart_disease', y='diabetes', ax=axes[1,0])
axes[1,0].set_title('Tỷ lệ tiểu đường theo heart_disease')

sm_order = df.groupby('smoking_history')['diabetes'].mean().sort_values(ascending=False).index
sns.barplot(data=df, x='smoking_history', y='diabetes', order=sm_order, ax=axes[1,1])
axes[1,1].set_title('Tỷ lệ tiểu đường theo smoking_history')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

### 5.4. Boxplot: biến số theo target

In [15]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x='diabetes', y=col, ax=ax)
    ax.set_title(f'{col} theo diabetes')
    ax.set_xticklabels(['Không', 'Có'])
plt.tight_layout(); plt.show()

/tmp/ipykernel_339433/3505440389.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(['Không', 'Có'])
/tmp/ipykernel_339433/3505440389.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(['Không', 'Có'])
/tmp/ipykernel_339433/3505440389.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(['Không', 'Có'])
/tmp/ipykernel_339433/3505440389.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(['Không', 'Có'])


## 6. Chuẩn bị dữ liệu cho mô hình

### 6.1. Tách features và target

In [16]:
X = df.drop(columns=['diabetes'])
y = df['diabetes']
print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nCác cột feature:')
print(list(X.columns))

X shape: (96146, 8)
y shape: (96146,)

Các cột feature:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']


### 6.2. Mã hoá biến categorical

Sử dụng `LabelEncoder` cho `gender` và `smoking_history`. Để cho pipeline Deep Learning về sau, ta dùng OrdinalEncoder cho đầu vào số hoá. Các biến binary (`hypertension`, `heart_disease`) giữ nguyên.

In [17]:
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_smoking = LabelEncoder()

X['gender_enc'] = le_gender.fit_transform(X['gender'])
X['smoking_enc'] = le_smoking.fit_transform(X['smoking_history'])

# Drop cột gốc
X = X.drop(columns=['gender', 'smoking_history'])

print('Các cột sau khi mã hoá:', list(X.columns))
X.head()

Các cột sau khi mã hoá: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_enc', 'smoking_enc']


,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,gender_enc,smoking_enc
0,80.0,0,1,25.19,6.6,140,0,4
1,54.0,0,0,27.32,6.6,80,0,0
2,28.0,0,0,27.32,5.7,158,1,4
3,36.0,0,0,23.45,5.0,155,0,1
4,76.0,1,1,20.14,4.8,155,1,1


### 6.3. Kiểm tra outliers (z-score)

Ta kiểm tra xem có giá trị ngoại lai đáng kể không bằng z-score trên các biến số.

In [18]:
from scipy.stats import zscore

z = np.abs(zscore(X.select_dtypes(include=[np.number])))
outliers_mask = (z > 4).any(axis=1)
print(f'⚠️ Số dòng có z-score > 4 (outlier mạnh): {outliers_mask.sum()} ({outliers_mask.mean()*100:.2f}%)')

# Với dataset lớn (gần 100k), ta giữ nguyên outliers vì MLP/ML vẫn học được; 
# sẽ kiểm tra trong modeling nếu cần.

⚠️ Số dòng có z-score > 4 (outlier mạnh): 4256 (4.43%)


### 6.4. Xử lý mất cân bằng lớp (Class Imbalance)

Target có ~91% lớp 0 và ~9% lớp 1. Đây là mất cân bằng rõ rệt. Ta dùng **SMOTE** trên tập **train** để cân bằng, giữ nguyên validation/test là phân bố gốc (phản ánh thực tế).

In [19]:
from sklearn.model_selection import train_test_split

# Chia train/validation/test theo thứ tự 70/15/15
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, random_state=RANDOM_STATE, stratify=y_train_val)

print('Train size:', X_train.shape)
print('Validation size:', X_val.shape)
print('Test size:', X_test.shape)
print('\nPhân bố target:')
for name, yy in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f"{name}: 0={sum(yy==0)}, 1={sum(yy==1)} (tỷ lệ 1: {yy.mean()*100:.2f}%)")

Train size: (67302, 8)
Validation size: (14422, 8)
Test size: (14422, 8)

Phân bố target:
Train: 0=61364, 1=5938 (tỷ lệ 1: 8.82%)
Val: 0=13150, 1=1272 (tỷ lệ 1: 8.82%)
Test: 0=13150, 1=1272 (tỷ lệ 1: 8.82%)


### 6.5. Chuẩn hoá (StandardScaler)

Chuẩn hoá để các feature có mean=0, std=1 — cần thiết cho MLP và nhiều mô hình ML (hồi quy logistic, SVM).

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit trên TRAIN chỉ, để tránh data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Scaler fit trên train.')
print('X_train_scaled shape:', X_train_scaled.shape)
print('X_val_scaled shape:', X_val_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)

Scaler fit trên train.
X_train_scaled shape: (67302, 8)
X_val_scaled shape: (14422, 8)
X_test_scaled shape: (14422, 8)


### 6.6. (Tùy chọn) SMOTE trên tập train

Vì MLP sau này cần cân bằng lớp, ta áp dụng SMOTE trên tập train **đã chuẩn hoá**.

In [21]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('Sau SMOTE:')
print('  X_train_res shape:', X_train_res.shape)
print('  0:', sum(y_train_res==0), '| 1:', sum(y_train_res==1))
print('  Cân bằng:', np.mean(y_train_res==1)*100, '%')

Sau SMOTE:
  X_train_res shape: (122728, 8)
  0: 61364 | 1: 61364
  Cân bằng: 50.0 %


### 6.7. Lưu tiền xử lý để tái sử dụng

Lưu `scaler`, encoder, và các biến đã xử lý về models để dùng lại trong modeling notebook.

In [22]:
import joblib
import os

MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(scaler, os.path.join(MODEL_DIR, 'diabetes_scaler.pkl'))
joblib.dump(le_gender, os.path.join(MODEL_DIR, 'le_gender.pkl'))
joblib.dump(le_smoking, os.path.join(MODEL_DIR, 'le_smoking.pkl'))

# Lưu data đã xử lý dạng numpy/pkl để dùng sau
np.savez_compressed(os.path.join(MODEL_DIR, 'preprocessed_data.npz'),
                    X_train=X_train_scaled, y_train=y_train,
                    X_train_res=X_train_res, y_train_res=y_train_res,
                    X_val=X_val_scaled, y_val=y_val,
                    X_test=X_test_scaled, y_test=y_test)

print('✅ Đã lưu preprocessing (scaler, encoders, data) vào', MODEL_DIR)

✅ Đã lưu preprocessing (scaler, encoders, data) vào ../models


### 6.8. Lưu tên feature


In [23]:
feature_names = list(X.columns)
joblib.dump(feature_names, os.path.join(MODEL_DIR, 'feature_names.pkl'))
print('Feature names:', feature_names)

Feature names: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_enc', 'smoking_enc']


## 7. Kết luận

- Dataset có **100,000** dòng, 8 feature + 1 target `diabetes` (binary).
- Không có missing; **4-5% duplicates** đã được loại bỏ.
- **Mất cân bằng lớp** (91% vs 9%) — xử lý bằng SMOTE trên tập train.
- Đã mã hoá categorical (`gender`, `smoking_history`), chuẩn hoá bằng `StandardScaler`.
- Đã chia **Train (70%) / Validation (15%) / Test (15%)** theo stratify.
- Tương quan cao nhất với target: `HbA1c_level`, `blood_glucose_level`, `bmi`, `age`.

**Dữ liệu đã sẵn sàng cho các bước modeling.**